<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase20_portafolios_esquina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

# 1. Dos activos: peso de varianza mínima según la correlación
s1, s2 = 0.10, 0.20

def peso_optimo(rho):
    return (s2**2 - rho * s1 * s2) / (s1**2 + s2**2 - 2 * rho * s1 * s2)

def vol_cartera(w, rho):
    var = w**2 * s1**2 + (1 - w)**2 * s2**2 + 2 * w * (1 - w) * rho * s1 * s2
    return var**0.5

for rho in (0.0, 0.3, 0.5, 0.8):
    print("Correlación", rho, "-> peso del activo tranquilo:", round(peso_optimo(rho), 3))
print("Correlación umbral (vol baja entre vol alta):", s1 / s2)
w8 = peso_optimo(0.8)
print("Con correlación 0.8, sin restricción: peso", round(w8, 3), "volatilidad", round(vol_cartera(w8, 0.8), 4))
print("Con correlación 0.8, solo largo (esquina, peso 1): volatilidad", round(vol_cartera(1.0, 0.8), 4))

# 2. Funciones para N activos
def min_var(cov):
    x = np.linalg.solve(cov, np.ones(len(cov)))
    return x / x.sum()

def min_var_solo_largo(cov, tope=1.0):
    n = len(cov)
    c = cov / np.mean(np.diag(cov))          # mismo óptimo, mejor escala para el optimizador
    res = minimize(lambda w: w @ c @ w, np.ones(n) / n, method="SLSQP",
                   bounds=[(0, tope)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"ftol": 1e-14, "maxiter": 500})
    w = res.x.copy()
    w[np.abs(w) < 1e-6] = 0.0
    return w / w.sum()

# 3. Tres activos de ejemplo
vol3 = np.array([0.10, 0.15, 0.20])
corr3 = np.array([[1.0, 0.2, 0.8], [0.2, 1.0, 0.3], [0.8, 0.3, 1.0]])
cov3 = np.outer(vol3, vol3) * corr3
w_libre, w_largo = min_var(cov3), min_var_solo_largo(cov3)
print("\nTres activos, sin restricción:", w_libre.round(3), "volatilidad", round(float((w_libre @ cov3 @ w_libre)**0.5), 4))
print("Tres activos, solo largo:     ", w_largo.round(3), "volatilidad", round(float((w_largo @ cov3 @ w_largo)**0.5), 4))
marg = cov3 @ w_largo
print("Covarianza de cada activo con la cartera solo largo:", marg.round(5), "| varianza de la cartera:", round(float(w_largo @ cov3 @ w_largo), 5))

# 4. Datos reales: seis pares contra el dólar
pares = {"USD/MXN": ("MXN=X", False), "USD/JPY": ("JPY=X", False), "USD/EUR": ("EURUSD=X", True),
         "USD/GBP": ("GBPUSD=X", True), "USD/CAD": ("CAD=X", False), "USD/BRL": ("BRL=X", False)}
precios = {}
for nombre, (ticker, invertir) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
    precios[nombre] = 1 / p if invertir else p       # EUR y GBP vienen como dólares por moneda: se invierten
rend = pd.DataFrame(precios).dropna().pct_change().dropna()
corte = int(len(rend) * 0.7)
entren, prueba = rend.iloc[:corte], rend.iloc[corte:]
cov_e = entren.cov().values
n = len(rend.columns)
print("\nDías:", len(rend), "| entrenamiento:", len(entren), "| prueba:", len(prueba))

carteras = {
    "Pesos iguales": np.ones(n) / n,
    "Inverso de la volatilidad": (1 / entren.std().values) / (1 / entren.std().values).sum(),
    "Mínima varianza libre": min_var(cov_e),
    "Mínima varianza solo largo": min_var_solo_largo(cov_e),
    "Solo largo, tope 40%": min_var_solo_largo(cov_e, tope=0.40),
}
pesos = pd.DataFrame(carteras, index=rend.columns)
print(pesos.round(3).to_string())
print("Pesos en cero (solo largo):", int((pesos["Mínima varianza solo largo"] < 1e-4).sum()), "de", n)

filas = []
for nombre, w in carteras.items():
    vol_in = float((w @ cov_e @ w)**0.5 * np.sqrt(252))
    vol_out = float((prueba.values @ w).std() * np.sqrt(252))
    filas.append({"cartera": nombre, "vol_dentro": vol_in, "vol_fuera": vol_out, "cociente": vol_out / vol_in})
print(pd.DataFrame(filas).round(3).to_string(index=False))

Correlación 0.0 -> peso del activo tranquilo: 0.8
Correlación 0.3 -> peso del activo tranquilo: 0.895
Correlación 0.5 -> peso del activo tranquilo: 1.0
Correlación 0.8 -> peso del activo tranquilo: 1.333
Correlación umbral (vol baja entre vol alta): 0.5
Con correlación 0.8, sin restricción: peso 1.333 volatilidad 0.0894
Con correlación 0.8, solo largo (esquina, peso 1): volatilidad 0.1

Tres activos, sin restricción: [ 1.069  0.264 -0.333] volatilidad 0.0784
Tres activos, solo largo:      [0.736 0.264 0.   ] volatilidad 0.0903
Covarianza de cada activo con la cartera solo largo: [0.00815 0.00815 0.01415] | varianza de la cartera: 0.00815


/tmp/ipykernel_6680/2874315788.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_6680/2874315788.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_6680/2874315788.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_6680/2874315788.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_6680/2874315788.py:54: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker


Días: 1570 | entrenamiento: 1099 | prueba: 471
         Pesos iguales  Inverso de la volatilidad  Mínima varianza libre  Mínima varianza solo largo  Solo largo, tope 40%
USD/MXN          0.167                      0.135                  0.081                       0.073                 0.090
USD/JPY          0.167                      0.162                  0.217                       0.206                 0.218
USD/EUR          0.167                      0.206                  0.317                       0.235                 0.287
USD/GBP          0.167                      0.174                 -0.129                       0.000                 0.000
USD/CAD          0.167                      0.224                  0.519                       0.485                 0.400
USD/BRL          0.167                      0.100                 -0.005                       0.000                 0.004
Pesos en cero (solo largo): 2 de 6
                   cartera  vol_dentro  vol_fuera  cocie